In [1]:
%pip install torch pandas numpy scikit-learn

  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
    --------------------------------------- 1.6/124.1 MB 11.1 MB/s eta 0:00:11
   - -------------------------------------- 5.2/124.1 MB 14.6 MB/s eta 0:00:09
   -- ------------------------------------- 8.4/124.1 MB 15.0 MB/s eta 0:00:08
   --- ------------------------------------ 11.8/124.1 MB 15.5 MB/s eta 0:00


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

DATA_DIR = Path(r"C:\Users\jhadz\Downloads\popbob")

SEED = 42
SKILLS = [f"S{i}" for i in range(1, 8)]
SKILL_TO_IDX = {skill: i for i, skill in enumerate(SKILLS)}
K = len(SKILLS)

HIDDEN = 100
BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Using device:", DEVICE)

def read_data(path):
    df = pd.read_csv(
        path,
        dtype={
            "user_id": str,
            "skill_code": str,
            "time_done": "int64",
            "y": "int64",
        },
    )

    assert set(df.columns) == {
        "user_id", "skill_code", "time_done", "y"
    }, f"Unexpected columns in {path}"

    assert df.notna().all().all(), f"Missing values in {path}"
    assert df["skill_code"].isin(SKILLS).all(), f"Unknown skill in {path}"
    assert df["y"].isin([0, 1]).all(), f"Invalid y in {path}"

    return df.sort_values(
        ["user_id", "time_done"],
        kind="mergesort",
    ).reset_index(drop=True)

def make_sequences(df):
    sequences = []

    for _, group in df.groupby("user_id", sort=False):
        skills = torch.tensor(
            group["skill_code"].map(SKILL_TO_IDX).to_numpy(),
            dtype=torch.long,
        )
        answers = torch.tensor(
            group["y"].to_numpy(),
            dtype=torch.long,
        )
        sequences.append((skills, answers))

    return sequences

def collate_sequences(batch):
    lengths = torch.tensor([len(skills) for skills, _ in batch])
    max_length = int(lengths.max())
    batch_size = len(batch)

    # 14 inputs = 7 skills × 2 previous-answer outcomes.
    x = torch.zeros(batch_size, max_length, 2 * K)
    target_skills = torch.zeros(
        batch_size, max_length, dtype=torch.long
    )
    target_y = torch.zeros(batch_size, max_length)

    mask = (
        torch.arange(max_length)[None, :]
        < lengths[:, None]
    )

    for i, (skills, answers) in enumerate(batch):
        n = len(skills)
        target_skills[i, :n] = skills
        target_y[i, :n] = answers.float()

        # At position t, give the model answer t-1.
        # Position 0 stays all zeros: no previous answer.
        if n > 1:
            previous = skills[:-1] + K * answers[:-1]
            x[i, torch.arange(1, n), previous] = 1.0

    return x, lengths, target_skills, target_y, mask

def make_loader(sequences, shuffle):
    return DataLoader(
        sequences,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_sequences,
    )

class DKT(nn.Module):
    def __init__(self, k=7, hidden=100):
        super().__init__()
        self.lstm = nn.LSTM(
            2 * k,
            hidden,
            batch_first=True,
        )
        self.output = nn.Linear(hidden, k)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=x.size(1),
        )
        return self.output(output)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_answers = 0

    for x, lengths, target_skills, target_y, mask in loader:
        x = x.to(DEVICE)
        target_skills = target_skills.to(DEVICE)
        target_y = target_y.to(DEVICE)
        mask = mask.to(DEVICE)

        with torch.set_grad_enabled(training):
            all_logits = model(x, lengths)
            chosen_logits = all_logits.gather(
                2,
                target_skills.unsqueeze(-1),
            ).squeeze(-1)

            answer_losses = F.binary_cross_entropy_with_logits(
                chosen_logits,
                target_y,
                reduction="none",
            )
            loss_sum = (answer_losses * mask).sum()
            loss = loss_sum / mask.sum()

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    5.0,
                )
                optimizer.step()

        total_loss += loss_sum.item()
        total_answers += mask.sum().item()

    return total_loss / total_answers

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_y = []
    all_probability = []
    all_skill = []

    for x, lengths, target_skills, target_y, mask in loader:
        logits = model(x.to(DEVICE), lengths)
        chosen_logits = logits.gather(
            2,
            target_skills.to(DEVICE).unsqueeze(-1),
        ).squeeze(-1)

        probability = torch.sigmoid(chosen_logits)
        mask = mask.to(DEVICE)

        all_y.append(
            target_y.to(DEVICE)[mask].cpu().numpy()
        )
        all_probability.append(
            probability[mask].cpu().numpy()
        )
        all_skill.append(
            target_skills.to(DEVICE)[mask].cpu().numpy()
        )

    return (
        np.concatenate(all_y).astype(int),
        np.concatenate(all_probability),
        np.concatenate(all_skill),
    )

def metric_rows(y, probability, skill_idx):
    rows = []

    groups = [
        ("overall", np.ones(len(y), dtype=bool))
    ]
    groups += [
        (skill, skill_idx == i)
        for i, skill in enumerate(SKILLS)
    ]

    for name, selected in groups:
        actual = y[selected]
        predicted = probability[selected]

        if len(actual) == 0:
            continue

        rows.append({
            "skill": name,
            "n_answers": len(actual),
            "correct_rate": actual.mean(),
            "accuracy_at_0.5": accuracy_score(
                actual,
                predicted >= 0.5,
            ),
            "majority_baseline_accuracy": max(
                actual.mean(),
                1 - actual.mean(),
            ),
            "auc": (
                roc_auc_score(actual, predicted)
                if len(np.unique(actual)) == 2
                else np.nan
            ),
            "brier": brier_score_loss(
                actual,
                predicted,
            ),
        })

    return pd.DataFrame(rows)

# Open train.csv only. Do not open test.csv yet.
train_df = read_data(DATA_DIR / "train.csv")
student_ids = train_df["user_id"].unique()

fit_ids, validation_ids = train_test_split(
    student_ids,
    test_size=0.10,
    random_state=SEED,
)

fit_df = train_df[
    train_df["user_id"].isin(fit_ids)
]
validation_df = train_df[
    train_df["user_id"].isin(validation_ids)
]

fit_loader = make_loader(
    make_sequences(fit_df),
    shuffle=True,
)
validation_loader = make_loader(
    make_sequences(validation_df),
    shuffle=False,
)

print("Total training rows:", len(train_df))
print("Training students:", len(fit_ids))
print("Validation students:", len(validation_ids))

Using device: cpu
Total training rows: 80000
Training students: 1796
Validation students: 200


In [3]:
# First, find a suitable number of epochs using validation students.
model = DKT(K, HIDDEN).to(DEVICE)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

best_validation_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(
        model,
        fit_loader,
        optimizer,
    )
    validation_loss = run_epoch(
        model,
        validation_loader,
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"validation loss: {validation_loss:.4f}"
    )

    if validation_loss < best_validation_loss - 1e-4:
        best_validation_loss = validation_loss
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

print("Selected epochs:", best_epoch)

# Now train a fresh final model using ALL rows in train.csv.
set_seed(SEED)

final_model = DKT(K, HIDDEN).to(DEVICE)
final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LEARNING_RATE,
)
full_train_loader = make_loader(
    make_sequences(train_df),
    shuffle=True,
)

for epoch in range(1, best_epoch + 1):
    loss = run_epoch(
        final_model,
        full_train_loader,
        final_optimizer,
    )
    print(
        f"Final training epoch {epoch:02d} | "
        f"loss: {loss:.4f}"
    )

print("Training complete.")

Epoch 01 | train loss: 0.5423 | validation loss: 0.4834
Epoch 02 | train loss: 0.4843 | validation loss: 0.4764
Epoch 03 | train loss: 0.4738 | validation loss: 0.4653
Epoch 04 | train loss: 0.4606 | validation loss: 0.4595
Epoch 05 | train loss: 0.4521 | validation loss: 0.4552
Epoch 06 | train loss: 0.4491 | validation loss: 0.4545
Epoch 07 | train loss: 0.4469 | validation loss: 0.4541
Epoch 08 | train loss: 0.4442 | validation loss: 0.4519
Epoch 09 | train loss: 0.4452 | validation loss: 0.4528
Epoch 10 | train loss: 0.4430 | validation loss: 0.4502
Epoch 11 | train loss: 0.4420 | validation loss: 0.4502
Epoch 12 | train loss: 0.4426 | validation loss: 0.4509
Epoch 13 | train loss: 0.4415 | validation loss: 0.4527
Epoch 14 | train loss: 0.4422 | validation loss: 0.4500
Epoch 15 | train loss: 0.4405 | validation loss: 0.4493
Epoch 16 | train loss: 0.4403 | validation loss: 0.4481
Epoch 17 | train loss: 0.4397 | validation loss: 0.4513
Epoch 18 | train loss: 0.4408 | validation loss:

In [4]:
from sklearn.metrics import confusion_matrix

test_df = read_data(DATA_DIR / "test.csv")

assert set(train_df["user_id"]).isdisjoint(
    set(test_df["user_id"])
), "A student appears in both train and test!"

test_loader = make_loader(
    make_sequences(test_df),
    shuffle=False,
)

test_y, test_probability, test_skill = collect_predictions(
    final_model,
    test_loader,
)

assert len(test_y) == len(test_df)

results = metric_rows(
    test_y,
    test_probability,
    test_skill,
)

print("TEST RESULTS")
print(results.round(4).to_string(index=False))

print("\nOverall confusion matrix at the 0.5 cutoff:")
print("Rows = actual 0/1; columns = predicted 0/1")
print(
    confusion_matrix(
        test_y,
        (test_probability >= 0.5).astype(int),
    )
)

TEST RESULTS
  skill  n_answers  correct_rate  accuracy_at_0.5  majority_baseline_accuracy    auc  brier
overall      20000        0.7833           0.7928                      0.7833 0.7371 0.1478
     S1       3550        0.8651           0.8662                      0.8651 0.7268 0.1077
     S2       9313        0.7535           0.7558                      0.7535 0.6961 0.1690
     S3        705        0.8794           0.8993                      0.8794 0.7478 0.0903
     S4       2237        0.6723           0.7255                      0.6723 0.7160 0.1883
     S5       2101        0.9229           0.9229                      0.9229 0.6998 0.0669
     S6       1411        0.6775           0.6931                      0.6775 0.6746 0.2004
     S7        683        0.8184           0.8302                      0.8184 0.6877 0.1341

Overall confusion matrix at the 0.5 cutoff:
Rows = actual 0/1; columns = predicted 0/1
[[  629  3705]
 [  440 15226]]


In [5]:
state_cpu = {
    name: value.detach().cpu().clone()
    for name, value in final_model.state_dict().items()
}

metadata = {
    "skills": SKILLS,
    "hidden": HIDDEN,
    "input_size": 2 * K,
    "selected_epochs": best_epoch,
    "label_rule": (
        "y=1 means correct on first attempt without hint"
    ),
    "first_answer_input": "all-zero 14-value vector",
}

pt_path = DATA_DIR / "dkt.pt"
pkl_path = DATA_DIR / "dkt_model.pkl"
metrics_path = DATA_DIR / "dkt_test_metrics.csv"

torch.save(
    {
        "state_dict": state_cpu,
        "metadata": metadata,
    },
    pt_path,
)

with open(pkl_path, "wb") as file:
    pickle.dump(
        {
            "state_dict": {
                name: value.numpy()
                for name, value in state_cpu.items()
            },
            "metadata": metadata,
        },
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

results.to_csv(metrics_path, index=False)

print("Saved:", pt_path)
print("Saved:", pkl_path)
print("Saved:", metrics_path)

Saved: C:\Users\jhadz\Downloads\popbob\dkt.pt
Saved: C:\Users\jhadz\Downloads\popbob\dkt_model.pkl
Saved: C:\Users\jhadz\Downloads\popbob\dkt_test_metrics.csv
